In [0]:
%run ../utils/init.py


In [0]:
df_bronze = spark.read.format("delta").load(f"{BRONZE}/dpe/")
print(f"Lignes bronze : {df_bronze.count()}")
df_bronze.printSchema()

In [0]:
df_silver = (df_bronze
    # Garder uniquement appartement et maison
    .filter(F.col("type_batiment").isin(["appartement", "maison"]))
    # Filtrer DPE récents uniquement (nouvelle méthode de calcul depuis 2021)
    .filter(F.col("annee_dpe") >= 2021)
    # Supprimer sans coordonnées
    .filter(F.col("latitude").isNotNull() & F.col("longitude").isNotNull())
    # Supprimer consommations aberrantes
    .filter(F.col("conso_energie_kwh_m2").between(0, 1500))
    # Déduplication
    .dropDuplicates(["id_dpe"])
    # Renommage
    .withColumnRenamed("nom_commune", "city")
    .withColumnRenamed("type_batiment", "property_type")
    .withColumnRenamed("conso_energie_kwh_m2", "energy_consumption_kwh_m2")
    .withColumnRenamed("emission_ges_kg_m2", "ges_emission_kg_m2")
    .withColumnRenamed("surface_m2", "building_area")
    # Sélection finale
    .select(
        "id_dpe", "date_dpe", "annee_dpe",
        "code_commune", "city", "code_departement",
        "property_type", "periode_construction",
        "etiquette_dpe", "etiquette_ges", "score_dpe_num",
        "building_area", "energy_consumption_kwh_m2", "ges_emission_kg_m2",
        "longitude", "latitude"
    )
)

print(f"Lignes silver : {df_silver.count()}")
df_silver.show(3)

In [0]:
df_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .partitionBy("annee_dpe", "code_departement") \
    .save(f"{SILVER}/dpe/")

print("✅ Silver DPE écrit")